In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.close('all')

# ROOT is the repo root, one level above the notebooks/ directory.
ROOT = Path.cwd().parent

# ── Load dataset ───────────────────────────────────────────────────────────────
df = pd.read_csv(ROOT / "data" / "raw" / "fraud_detection.csv")
print(f"Full dataset shape: {df.shape}")

# ── Sample for tractable EDA ───────────────────────────────────────────────────
df_sample = df.sample(500_000, random_state=42)
print(f"Sample shape: {df_sample.shape}")

# ── Basic checks ───────────────────────────────────────────────────────────────
print(df_sample.info())
print(df_sample.describe())
print("\nMissing values:\n", df_sample.isnull().sum())

# ── Target distribution ────────────────────────────────────────────────────────
print("\nFraud value counts:\n", df_sample['isFraud'].value_counts())
print("\nFraud percentage:\n", df_sample['isFraud'].value_counts(normalize=True) * 100)

# ── Visualise target ───────────────────────────────────────────────────────────
plt.figure(figsize=(6, 4))
sns.countplot(x='isFraud', data=df_sample)
plt.title("Fraud Distribution")
plt.show()

# ── Transaction types ──────────────────────────────────────────────────────────
print("\nTransaction type counts:\n", df_sample['type'].value_counts())

plt.figure(figsize=(8, 5))
sns.countplot(x='type', data=df_sample, order=df_sample['type'].value_counts().index)
plt.title("Transaction Types")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ── Fraud by type ──────────────────────────────────────────────────────────────
# Only TRANSFER and CASH_OUT carry fraud in the PaySim dataset.
fraud_by_type = pd.crosstab(df_sample['type'], df_sample['isFraud'])
print("\nFraud by transaction type:\n", fraud_by_type)

fraud_by_type.plot(kind='bar', figsize=(10, 6))
plt.title("Fraud Count by Transaction Type")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# ── Amount distribution ────────────────────────────────────────────────────────
print("\nAmount stats:\n", df_sample['amount'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(df_sample['amount'], bins=50)
plt.title("Transaction Amount Distribution")
plt.show()

# ── Fraud transactions ─────────────────────────────────────────────────────────
fraud_txns = df_sample[df_sample['isFraud'] == 1]
print("\nFraud transaction amount stats:\n", fraud_txns['amount'].describe())

# ── Correlation heatmap ────────────────────────────────────────────────────────
numeric_df = df_sample.select_dtypes(include=['number'])
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f')
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# ── Duplicate check ────────────────────────────────────────────────────────────
print(f"\nDuplicate rows: {df_sample.duplicated().sum()}")

# ── Save sample ───────────────────────────────────────────────────────────────
out_path = ROOT / "data" / "processed" / "fraud_sample.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_sample.to_csv(out_path, index=False)
print(f"Sample saved to {out_path}")